Run design pipeline

This notebook is a thin wrapper around `run_design_pipeline.py`. It lets you configure the input PDB folder and key parameters, then executes the full pipeline end-to-end.


In [1]:
#@title Setup (~13mins)
%%capture
structure_scoring = "Boltz"  # @param ['Boltz', 'Boltz_template', 'ProtenixScore']

if structure_scoring == "ProtenixScore":
  !git clone https://github.com/cytokineking/ProtenixScore
  %cd ProtenixScore
  !./install_protenixscore.sh --no-conda
  %cd /content
  !pip install https://data.pyg.org/whl/torch-2.7.0%2Bcu126/torch_cluster-1.6.3%2Bpt27cu126-cp312-cp312-linux_x86_64.whl
if structure_scoring == "Boltz" or structure_scoring == "Boltz_template":
  !pip install virtualenv
  !virtualenv boltz_env
  !source /content/boltz_env/bin/activate; pip install boltz[cuda] -U
  !pip uninstall torch torchvision torchaudio -y
  !pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
  !pip install https://data.pyg.org/whl/torch-2.8.0%2Bcu128/torch_cluster-1.6.3%2Bpt28cu128-cp312-cp312-linux_x86_64.whl

!pip install torch_geometric
!pip install pyrosetta-installer
!python -c 'import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()'
!pip install biopython
!pip install tqdm
!pip install dm-tree
!pip install biotite
!pip install pyyaml
!pip install modelcif
!pip install e3nn
#!pip install easydict


!git clone https://github.com/nedru004/MPNN_affinity_maturation.git
#working_dir = "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation"
working_dir = "/content/MPNN_affinity_maturation"

!git clone https://github.com/dauparas/ProteinMPNN.git

!git clone https://gitlab.com/mjslee0921/flowpacker
#%pip install --use-deprecated=legacy-resolver -r flowpacker/requirements.txt

#!git clone https://github.com/ohuelab/boltzina.git
#!https://github.com/nedru004/boltz_score.git

import sys
sys.path.insert(0, working_dir)
sys.path.insert(0, "/content/flowpacker")

# Run design pipeline

This notebook runs the full affinity-maturation pipeline via `run_design_pipeline.py`. You can run **iteration 1** with explicit fixed positions, then **iteration 2+** by passing the previous run's `fixed_residue_pyrosetta.csv` and using `merge_motif_pdb` as the next input — that closes the loop so MPNN fixes the interface residues identified in the prior round.


In [ ]:
import os
from pathlib import Path
from run_design_pipeline import run_pipeline, get_fixed_positions_from_csv

working_dir = Path(os.getcwd())  # in Colab: /content/MPNN_affinity_maturation
test_dir = working_dir / "test"

# --- Iteration 1: initial PDBs, explicit fixed positions ---
input_pdb_dir = working_dir / "test"  # folder with your starting PDBs
fixed_positions = "11 12 13 14 15"   # positions to fix in chain A for first round

# Scoring method: "boltz", "boltz_template", or "protenixscore"
scoring_method = "boltz"

merge_dir = run_pipeline(
    working_dir=working_dir,
    input_pdb_dir=input_pdb_dir,
    chains_to_design="A",
    num_seq_per_target=8,
    sampling_temp=0.5,
    seed=37,
    fixed_positions=fixed_positions,
    fixed_positions_csv=None,  # first run: use fixed_positions above
    scoring_method=scoring_method,
    binder_chain="A",
    target_chain="M",
    interface_dist=10.0,
    energy_threshold=-5.0,
)
print("Iteration 1 done. Merged motif PDBs in:", merge_dir)


## Iteration 2 (closed loop)

Use the previous run's merged motif PDBs as input and derive fixed positions from the key residues identified by PyRosetta. Run this cell after iteration 1 has finished.

In [ ]:
# Input: previous run's merge_motif_pdb and fixed_residue_pyrosetta.csv
prev_fixed_csv = test_dir / "fixed_residue_pyrosetta.csv"
prev_merge_dir = test_dir / "merge_motif_pdb"

# Optional: inspect which positions will be fixed in this round
if prev_fixed_csv.exists():
    next_fixed = get_fixed_positions_from_csv(prev_fixed_csv)
    print("Fixed positions for this round (from previous key_res):", next_fixed)

merge_dir_2 = run_pipeline(
    working_dir=working_dir,
    input_pdb_dir=prev_merge_dir,
    chains_to_design="A",
    num_seq_per_target=8,
    sampling_temp=0.5,
    seed=37,
    fixed_positions="",  # ignored when fixed_positions_csv is set
    fixed_positions_csv=prev_fixed_csv,  # closes the loop
    scoring_method=scoring_method,  # same as iteration 1, or change to "boltz_template" / "protenixscore"
    binder_chain="A",
    target_chain="M",
    interface_dist=10.0,
    energy_threshold=-5.0,
)
print("Iteration 2 done. Merged motif PDBs in:", merge_dir_2)